In [ ]:
import pandas as pd

# Path to data frame with WSIs
# df_path = r"D:\DATA\with_snomed_category.csv"
# df_path = r"D:\DATA\abmil_exp2.csv"
# df_path = r"D:\DATA\abmil_exp3.csv"
df_path = r"D:\DATA\abmil_inference_exp3.csv"

df_all = pd.read_csv(df_path)
print(df_all.columns)

# Paths for ABMIL inference output
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"
checkpoint_path = r"D:\NOTEBOOKS\Christine\checkpoints\exp3_h-optimus-0\fold_1_auc_0.8883.pt"
cache_path = r"D:\NOTEBOOKS\Christine\checkpoints\exp3_h-optimus-0\fold_1_inference_cache.pkl"

In [ ]:
from abmil_pipeline import DiseaseClassification

all_filenames = df_all['filename'].tolist()

classifier = DiseaseClassification(checkpoint_path=checkpoint_path, zarr_dir=zarr_dir, slides=all_filenames, cache_path = cache_path)
print('Classifier initialized for', len(all_filenames), 'slides')

In [ ]:
# Prepare true labels if available
true_labels = df_all.get('M_idx', None)
if true_labels is not None:
    true_labels = true_labels.tolist()

classifier.process_slides(true_labels=true_labels, top_k=100)

In [ ]:
# Get assessment report + confusion matrix
report = classifier.assessment_report(true_labels=true_labels)

In [ ]:
# ROI-to-LMD setup
# Use the inferred top tiles from one slide as ROIs.
# Edit slide_index or slide_filename below to choose which slide to export.
# In the next cell, open Napari, add a `calibration_points` points layer and a `polygons` shape layer, then close the viewer.

from pathlib import Path
import os

import geopandas as gpd
import numpy as np
import spatialdata as sd
from shapely.geometry import Polygon
from spatialdata.models import ShapesModel

slide_index = 0
slide_filename = None  # Example: "some_slide.mrxs"

if slide_filename is not None:
    matching_indices = [
        idx for idx, slide_path in enumerate(classifier.slides)
        if os.path.basename(slide_path) == slide_filename or slide_path == slide_filename
    ]
    if not matching_indices:
        raise ValueError(f"Slide not found: {slide_filename}")
    slide_index = matching_indices[0]

n_rois = 20

slide_path = classifier.slides[slide_index]
slide_data = classifier._slide_cache[slide_path]
top_tiles_df = slide_data["top_tiles_df"].head(n_rois).copy()

tiles_gdf = gpd.GeoDataFrame(
    top_tiles_df[["tile_id", "attention", "geometry"]].copy(),
    geometry="geometry",
)

sdata = sd.SpatialData()
sdata.shapes["tiles"] = ShapesModel.parse(tiles_gdf)

roi_polygons = [
    np.array(geometry.exterior.coords)
    for geometry in tiles_gdf.geometry
    if geometry.geom_type == "Polygon"
]

In [ ]:
import napari

viewer = napari.Viewer()
viewer.add_shapes(
    roi_polygons,
    shape_type="polygon",
    edge_color="red",
    face_color="red",
    name="top_tiles_roi",
)
viewer.add_points(np.empty((0, 2)), name="calibration_points")
viewer.add_shapes([], shape_type="polygon", name="polygons")

# Use the ROI layer for reference, then draw the calibration points and polygons in Napari.
napari.run()

In [ ]:
from spatialdata.models import PointsModel

# After closing Napari, collect the manually created layers.
points_layer = viewer.layers["calibration_points"]
image_points = points_layer.data

if len(image_points) == 0:
    raise ValueError("Add at least one point to the calibration_points layer before continuing.")

sdata.points["calibration_points"] = PointsModel.parse(np.array(image_points))

square_layer = viewer.layers["polygons"]
image_square = square_layer.data

if len(image_square) == 0:
    raise ValueError("Add at least one polygon to the polygons layer before continuing.")

square_polygons = [Polygon(coords) for coords in image_square]
square_gdf = gpd.GeoDataFrame(
    {"shape_id": [f"square_{i}" for i in range(len(square_polygons))]},
    geometry=square_polygons,
)
sdata.shapes["square"] = ShapesModel.parse(square_gdf)

In [ ]:
import os

from dvpio.write import write_lmd

# Convert the manually created annotations into an LMD XML file.
slide_height = int(slide_data["tile_table"]["geometry"].bounds["maxy"].max()) + 1
path_lmd = os.path.join(Path(cache_path).parent, f"{Path(slide_path).stem}_top_tiles.xml")

affine_transformation = np.array([
    [1,  0, 0],
    [0, -1, slide_height],
    [0,  0, 1],
])

write_lmd(
    path_lmd,
    sdata.shapes["tiles"],
    calibration_points=sdata.points["calibration_points"],
    affine_transformation=affine_transformation,
)

print(f"Saved LMD file to {path_lmd}")